In [ ]:
# Install the pinecone library
!pip install pinecone
import pinecone
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.3/516.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.1/239.1 kB 13.5 MB/s eta 0:00:00


In [ ]:
pc = pinecone.Pinecone(api_key="REDACTED")
index1 = pc.Index("e5")
embedding_model = SentenceTransformer('intfloat/multilingual-e5-large-instruct')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [ ]:
def retrieve_context(query, top_k=10, min_score=0.75):
    # Encode the query using the embedding model (e.g., E5 or MiniLM)
    query_vector = embedding_model.encode(["query: " + query])[0].tolist()

    # Query Pinecone index
    response = index1.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    # Filter matches based on similarity score threshold
    high_quality_chunks = [
        match['metadata']['text']
        for match in response['matches']
        if match['score'] >= min_score
    ]

    return high_quality_chunks


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "teknium/OpenHermes-2.5-Mistral-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


tokenizer_config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
def generate_answer(query, context_chunks, max_context_words=400):
    context = ""
    word_count = 0
    for chunk in context_chunks:
        word_count += len(chunk.split())
        if word_count > max_context_words:
            break
        context += chunk + "\n\n"

    prompt = (
        "You are a helpful multilingual assistant. "
        "Use the context below to answer the question accurately. "
        "Respond in the same language as the question.\n\n"
        f"Question: {query}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

    result = generator(prompt, max_new_tokens=200, do_sample=False)
    return result[0]['generated_text']


In [ ]:
query = "Can mental health problems affect physical health?"
import textwrap
context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)

wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Question:", query)
print("Answer:\n", wrapped_answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Question: Can mental health problems affect physical health?
Answer:
 Yes, mental health problems can affect physical health. Mental health problems can lead to a variety
of physical health issues, such as increased risk of infant and child mortality, higher rates of
HIV/AIDS infection, and poor adherence to treatments for other ailments. Additionally, mental health
problems can also lead to unsafe sex and drug use, which can further impact physical health.


In [ ]:
query = "How does social media affect teenage mental health?"
import textwrap
context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)

wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Question:", query)
print("Answer:\n", wrapped_answer)

Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Question: How does social media affect teenage mental health?
Answer:
 Social media can have both positive and negative effects on teenage mental health. On one hand, it
can provide a sense of community and support for teenagers who may feel isolated or alone. It can
also be a platform for self-expression and creativity. On the other hand, social media can
contribute to feelings of anxiety, depression, and low self-esteem. Comparison to others on social
media can lead to feelings of inadequacy and the pressure to conform to unrealistic standards of
beauty and success. Cyberbullying is also a growing concern, with teenagers being targeted and
harassed online. It's important for teenagers to use social media in a healthy and balanced way, and
for parents and educators to educate them about the potential risks and benefits.


In [ ]:
query = "Why is mental health important for public health policy?"
import textwrap
context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)

wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Question:", query)
print("Answer:\n", wrapped_answer)


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Question: Why is mental health important for public health policy?
Answer:
 Mental health is important for public health policy because it affects various aspects of society
and has a significant impact on physical health targets, such as reducing infant and child mortality
through the treatment of postnatal depression. Improved mental health also reduces HIV/AIDS
infection rates among young people by decreasing unsafe sex and drug use. Additionally, better
mental health leads to better adherence to treatments for other ailments like tuberculosis,
HIV/AIDS, hypertension, diabetes, and cancer. Mental health is also crucial for caregivers,
employers, and governments, as it results in a lower burden of care, reduced absenteeism, and higher
productivity, ultimately leading to less cost-shifting and transfer payments. Furthermore, mental
health is a key variable in successful programs for sustainable development and poverty reduction.
Lack of comprehensive mental health policies and legisla

In [ ]:

query = "How does cognitive behavioral therapy (CBT) help with anxiety?"
import textwrap
context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)

wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Question:", query)
print("Answer:\n", wrapped_answer)

Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Question: How does cognitive behavioral therapy (CBT) help with anxiety?
Answer:
 Cognitive behavioral therapy (CBT) helps with anxiety by identifying and changing negative thought
patterns and behaviors that contribute to anxiety. CBT focuses on modifying maladaptive behaviors
that have been learned and reinforced, and it teaches individuals coping strategies and techniques
to manage their anxiety. By addressing the underlying thought processes and behaviors that
contribute to anxiety, CBT can help individuals develop more adaptive ways of thinking and
responding to stressors, ultimately reducing anxiety symptoms.


In [ ]:
import textwrap

#  Swedish query: What causes panic attacks?
query = "Vad orsakar panikattacker?"

context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)
wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Question:", query)
print("Answer:\n", wrapped_answer)


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Fråga: Vad orsakar panikattacker?
Svar:
 Panikattacker orsakas av psykologiska faktorer som bara kan upptäckas och behandlas av en
kvalificerad psykolog eller terapeut.    Vad orsakar panikattacker?


In [ ]:

import textwrap

#  Swedish query: When should someone seek professional help for mental health?
query = "När bör man söka professionell hjälp för psykisk ohälsa?"

context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)
wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Fråga:", query)
print("Svar:\n", wrapped_answer)


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Fråga: När bör man söka professionell hjälp för psykisk ohälsa?
Svar:
 När bör man söka professionell hjälp för psykisk ohälsa?  Man bör söka professionell hjälp för
psykisk ohälsa om man upplever att det påverkar sin vardag och sociala liv, eller om man upplever
att det påverkar sin arbetsförmåga och hälsa. Om man upplever att det blir svårt att hantera
känslor, tankar eller beteenden som är oroliga eller orättvisa, kan det också vara lämpligt att söka
hjälp. Det är viktigt att söka hjälp tidigt för att få bäst möjliga behandling och förbättra sin
hälsa.


In [ ]:
import textwrap

#  Swedish query: What role does sleep play in mental health?
query = "Vilken roll spelar sömn för psykisk hälsa?"

context_docs = retrieve_context(query, top_k=10, min_score=0.75)
answer = generate_answer(query, context_docs)
wrapped_answer = textwrap.fill(answer.split("Answer:")[-1].strip(), width=100)

print("Fråga:", query)
print("Svar:\n", wrapped_answer)


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Fråga: Vilken roll spelar sömn för psykisk hälsa?
Svar:
 Sömn spelar en viktig roll för psykisk hälsa.  In English: Sleep plays an important role in mental
health.
